In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, paths
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "india"
vehicle = "rice"
has_simulations = True
has_anemia_model = True
has_ntd_model = True


In [4]:
rescaled_pregnancy_root = paths.source_root(paths.RESCALED_PREGNANCY_RESULTS_ROOT, data_source)
rescaled_child_root = paths.source_root(paths.RESCALED_CHILD_RESULTS_ROOT, data_source)
anemia_root = paths.source_root(paths.NON_PREGNANT_ANEMIA_RESULTS_ROOT, data_source)
print(f"{data_source}:\n  {rescaled_pregnancy_root}\n  {rescaled_child_root}\n  {anemia_root}")

model2.6.3:
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/rescaled_pregnancy
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/rescaled_child
  /mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.3/non_pregnant_anemia


In [5]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [6]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/ylls.parquet"
if has_simulations:
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,141,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,141,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,141,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,141,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,zero,0,129,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,zero,0,129,0.0
539997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,zero,0,129,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,zero,0,129,0.0


In [7]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [8]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [9]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  421862.784599
                                  2                  252001.099240
                                  3                  299794.461672
                                  4                  288856.199864
                                  5                  175580.738132
intervention  maternal_disorders  1                  415159.773884
                                  2                  250254.477162
                                  3                  296392.551074
                                  4                  285455.502296
                                  5                  173775.503965
zero          maternal_disorders  1                  431171.980330
                                  2                  256605.660716
                                  3                  305054.203059
                                  4                  295526.150075
            

In [10]:
path = f"{rescaled_pregnancy_root}/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"{rescaled_pregnancy_root}/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,141,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,141,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,141,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,141,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,postpartum,95_plus,severe,5,zero,0,129,0.0
1889996,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,zero,0,129,0.0
1889997,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,zero,0,129,0.0
1889998,ylds,cause,all_causes,all_causes,95_plus,severe,5,zero,0,129,0.0


In [11]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [12]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   99409.156076
                                  2                   77687.067142
                                  3                   62298.484494
                                  4                   53221.693806
                                  5                   41165.849066
              maternal_disorders  1                   74011.458381
                                  2                   39269.018480
                                  3                   45781.819588
                                  4                   45564.187119
                                  5                   31814.621756
intervention  anemia              1                   95524.428624
                                  2                   74605.916267
                                  3                   59497.578924
                                  4                   50837.940201
            

In [13]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   99409.156076
                                  2                   77687.067142
                                  3                   62298.484494
                                  4                   53221.693806
                                  5                   41165.849066
              maternal_disorders  1                  495874.242980
                                  2                  291270.117720
                                  3                  345576.281260
                                  4                  334420.386983
                                  5                  207395.359888
intervention  anemia              1                   95524.428624
                                  2                   74605.916267
                                  3                   59497.578924
                                  4                   50837.940201
            

In [14]:
ylds_path = f"{rescaled_child_root}/{vehicle}/{location}/ylds.parquet"
if has_simulations:
    assert (pd.read_parquet(ylds_path)['value'] == 0).all()

In [15]:
path = f"{rescaled_child_root}/{vehicle}/{location}/ylls.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_cause_and_scenario below, so we'd
# need to change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_ylls = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    # Combos with no iron pathway (e.g. ethiopia/salt, nigeria/salt) have no
    # Vivarium sims, so the corresponding DALYs are 0
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"{rescaled_child_root}/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,41810.570637
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,32926.106035
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,28597.306682
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,22099.062794
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,22744.377339
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,3560.441002
23996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,2105.100203
23997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,3551.444476
23998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,2728.249651


In [16]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.791522e+07
                      2                  1.481069e+07
                      3                  1.313146e+07
                      4                  1.251249e+07
                      5                  1.185047e+07
intervention  lbwsg   1                  1.788726e+07
                      2                  1.479053e+07
                      3                  1.311563e+07
                      4                  1.249363e+07
                      5                  1.183877e+07
zero          lbwsg   1                  1.795077e+07
                      2                  1.483410e+07
                      3                  1.314924e+07
                      4                  1.253438e+07
                      5                  1.185806e+07
Name: value, dtype: float64

In [17]:
path = f"{anemia_root}/{vehicle}/{location}/ylds.parquet"
if has_anemia_model:
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"{anemia_root}/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1188.516847,zero
1,Female,0.0,0.019178,2,903.023359,zero
2,Female,0.0,0.019178,3,800.254117,zero
3,Female,0.0,0.019178,4,665.519035,zero
4,Female,0.0,0.019178,5,465.944230,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,224.414295,intervention
746,Male,95.0,125.000000,2,218.670007,intervention
747,Male,95.0,125.000000,3,224.967836,intervention
748,Male,95.0,125.000000,4,219.703724,intervention


In [18]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.867990e+06
                      2                  1.829611e+06
                      3                  1.867698e+06
                      4                  1.777556e+06
                      5                  1.612937e+06
intervention  anemia  1                  1.793659e+06
                      2                  1.752126e+06
                      3                  1.786409e+06
                      4                  1.701918e+06
                      5                  1.542404e+06
zero          anemia  1                  1.959876e+06
                      2                  1.917732e+06
                      3                  1.950274e+06
                      4                  1.851084e+06
                      5                  1.651151e+06
Name: value, dtype: float64

In [19]:
scenarios[1]

'zero'

In [20]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

np.float64(-397435.1234522294)

In [21]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  640972.659367
                      2                  523760.052168
                      3                  482118.659339
                      4                  411423.442962
                      5                  320668.074127
intervention  anemia  1                  617951.974540
                      2                  503921.213535
                      3                  464150.257727
                      4                  395953.709634
                      5                  308526.114053
zero          anemia  1                  666214.496462
                      2                  544512.291368
                      3                  499379.450266
                      4                  426086.827911
                      5                  327533.851693
Name: value, dtype: float64

In [22]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

np.float64(-84784.02973579802)

In [23]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.516687e+06
                      2                  4.102192e+06
                      3                  4.040514e+06
                      4                  3.717001e+06
                      5                  3.302007e+06
intervention  anemia  1                  4.334717e+06
                      2                  3.927662e+06
                      3                  3.865949e+06
                      4                  3.559506e+06
                      5                  3.160452e+06
zero          anemia  1                  4.739109e+06
                      2                  4.299769e+06
                      3                  4.218140e+06
                      4                  3.870166e+06
                      5                  3.378457e+06
Name: value, dtype: float64

In [24]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if has_ntd_model:
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  293733.701862
                      2                  259483.070661
                      3                  232954.812359
                      4                  219664.907598
                      5                  187079.380834
baseline      ntd     1                  283500.810798
                      2                  252136.566758
                      3                  226793.674249
                      4                  214285.603276
                      5                  185165.750831
intervention  ntd     1                  181387.974907
                      2                  169948.248237
                      3                  153260.307487
                      4                  147224.599142
                      5                  144832.182133
Name: value, dtype: float64

In [25]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.616097e+06
                                  2                  4.179879e+06
                                  3                  4.102812e+06
                                  4                  3.770223e+06
                                  5                  3.343173e+06
              lbwsg               1                  1.791522e+07
                                  2                  1.481069e+07
                                  3                  1.313146e+07
                                  4                  1.251249e+07
                                  5                  1.185047e+07
              maternal_disorders  1                  4.958742e+05
                                  2                  2.912701e+05
                                  3                  3.455763e+05
                                  4                  3.344204e+05
                          

In [26]:
import pathlib

In [27]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)